# Graph Neural Networks on spatial data

This notebook builds a Graph Neural Network (GNN) on acolon cancer slide. We turn the tissue into a graph (cells are the nodes, neighbouring cells are connected by edges) and let each cell exchange information with its neighbours. This is called *message passing*, and it is what a GNN does.

We use it for two tasks: predicting the Novae spatial domains (supervised), then finding the niches without any labels (unsupervised).

The idea to keep in mind throughout: a cell's expression tells us **what it is** (its cell type), while its neighbourhood tells us **where it lives** (its niche). The `novae_domains` we use as labels were computed by Novae, which is itself a GNN, so here we rebuild the intuition behind it.

> ℹ️ This is the **student** notebook: the exercise cells contain `...` for you to fill in. Everything else (data loading, plots) is already written. A solution notebook is available if you get stuck.

## Dependencies

We'll need `novae`, `torch` and `torch-geometric`. Use the `Python (gnn_tutorial)` kernel: Novae needs Python ≥ 3.11.

In [ ]:
# !pip install "novae>=1.1" pyarrow scanpy scikit-learn torch torch-geometric matplotlib

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from copy import deepcopy

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

import scanpy as sc
import novae

from sklearn.neighbors import kneighbors_graph
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, scatter, degree
from torch_geometric.nn import SAGEConv, GCNConv

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

## Loading the slide

We use `novae.load_dataset` to download a public colon cancer slide. With `annotations=True` it also attaches the pre-computed Novae spatial domains.

> ℹ️ We use `[0]` because `load_dataset` returns a list of slides, and we selected a single one by name. The first call downloads about 400 MB and caches it.

In [ ]:
adata = novae.load_dataset(
    pattern="Xenium_V1_Human_Colon_Cancer_P5_CRC_Add_on_FFPE_outs",
    annotations=True,
)[0]
adata

The elements we'll use are:

1. `.X`: log-normalised expression of the 422-gene Xenium panel (our node features)
2. `.obsm["spatial"]`: the cell coordinates (used to build the graph)
3. `.obs["novae_domains_res1"]`: the Novae spatial domains, 10 niches (our target)
4. `.obs["cell_type"]`: coarse cell types

The full slide has about 276k cells. We crop a contiguous square (~50k cells) to keep things fast. A contiguous crop keeps each cell's neighbourhood intact, which a random subsample would not.

In [ ]:
sp = adata.obsm["spatial"]
box = (2000, 4600, 2000, 4600)                        # x_min, x_max, y_min, y_max (microns)
keep = (sp[:, 0] >= box[0]) & (sp[:, 0] < box[1]) & (sp[:, 1] >= box[2]) & (sp[:, 1] < box[3])
ad = adata[keep].copy()
ad = ad[~ad.obs["novae_domains_res1"].isna()].copy()  # a few cells are unassigned
for col in ["novae_domains_res1", "cell_type"]:
    ad.obs[col] = ad.obs[col].astype("category").cat.remove_unused_categories()

N = ad.n_obs
coords = np.asarray(ad.obsm["spatial"], dtype="float32")
expr   = np.asarray(ad.X.todense() if hasattr(ad.X, "todense") else ad.X, dtype="float32")
DOMAINS = list(ad.obs["novae_domains_res1"].cat.categories)
CTYPES  = list(ad.obs["cell_type"].cat.categories)
y_domain   = ad.obs["novae_domains_res1"].cat.codes.to_numpy()
y_celltype = ad.obs["cell_type"].cat.codes.to_numpy()
D, n_genes = len(DOMAINS), expr.shape[1]
print(f"{N} cells, {n_genes} genes, {D} domains, {len(CTYPES)} cell types")

## Looking at the tissue

Let's colour the cells by cell type and by Novae domain. Cell types are found everywhere, while domains form large contiguous regions.

In [ ]:
DOMAIN_COLORS = np.array(plt.cm.tab10.colors)[:D]
TYPE_COLORS   = np.array([[0.84,0.19,0.15],[0.17,0.44,0.71],[0.30,0.69,0.29],[0.60,0.40,0.80]])[:len(CTYPES)]

def plot_spatial(ax, labels, names, colors, title, s=4):
    for i in np.argsort(np.bincount(labels))[::-1]:
        m = labels == i
        ax.scatter(coords[m,0], coords[m,1], s=s, color=colors[i], label=names[i], linewidths=0)
    ax.set(title=title, xticks=[], yticks=[], aspect="equal"); ax.invert_yaxis()

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
plot_spatial(axes[0], y_celltype, CTYPES, TYPE_COLORS, "cell type")
plot_spatial(axes[1], y_domain, DOMAINS, DOMAIN_COLORS, "Novae spatial domain")
for ax in axes: ax.legend(markerscale=3, fontsize=8, loc="upper left", bbox_to_anchor=(1,1))
plt.tight_layout(); plt.show()

Now a quick check. We run a PCA on the expression and colour it both ways. The expression separates the cell types, but not the domains: different domains are made of the same cell types, so they overlap in expression space. A model that only sees a cell's own expression can guess its type, but not its niche.

In [ ]:
pcs = PCA(n_components=2, random_state=SEED).fit_transform(StandardScaler().fit_transform(expr))
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for i in range(len(CTYPES)):
    m = y_celltype == i; axes[0].scatter(pcs[m,0], pcs[m,1], s=4, color=TYPE_COLORS[i], label=CTYPES[i], linewidths=0)
for i in range(D):
    m = y_domain == i;   axes[1].scatter(pcs[m,0], pcs[m,1], s=4, color=DOMAIN_COLORS[i], label=DOMAINS[i], linewidths=0)
axes[0].set_title("PCA coloured by cell type"); axes[1].set_title("PCA coloured by domain")
for ax in axes: ax.set(xlabel="PC1", ylabel="PC2"); ax.legend(markerscale=3, fontsize=7)
plt.tight_layout(); plt.show()

## Building a spatial graph

We connect each cell to its nearest spatial neighbours. In PyTorch Geometric, a graph is stored as an `edge_index` of shape `[2, num_edges]`, where each column `[i, j]` is an edge from node `i` to node `j`.

### Exercice 1: k-nearest-neighbours graph

1. Build a symmetric k-NN `edge_index` from the spatial `coords` (not from the expression).

_Hint: use `kneighbors_graph(coords, n_neighbors=K_GRAPH, mode="connectivity", include_self=False)` from `sklearn`, take `.tocoo()`, stack `.row` and `.col` into a `torch.long` tensor, and pass it through `to_undirected`._

In [ ]:
K_GRAPH = 8

# Exercice 1: build a symmetric k-NN edge_index from coords
edge_index = ...

deg = degree(edge_index[0], num_nodes=N)
print(f"{edge_index.shape[1]} edges, average degree {deg.mean():.1f}")

We can visualise the graph on a zoom-in, coloured by domain.

In [ ]:
zx0, zx1, zy0, zy1 = 2800, 3300, 2800, 3300
zm = (coords[:,0]>=zx0)&(coords[:,0]<zx1)&(coords[:,1]>=zy0)&(coords[:,1]<zy1)
zset = set(np.where(zm)[0].tolist()); src, dst = edge_index.numpy()
emask = np.array([s in zset and d in zset for s, d in zip(src, dst)])
fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.add_collection(LineCollection(coords[edge_index.numpy().T[emask]], colors="0.6", linewidths=0.4, alpha=0.6, zorder=0))
for i in range(D):
    m = zm & (y_domain == i)
    if m.sum(): ax.scatter(coords[m,0], coords[m,1], s=30, color=DOMAIN_COLORS[i], label=DOMAINS[i], zorder=1, linewidths=0)
ax.set(title=f"k-NN graph (k={K_GRAPH}), zoom", xticks=[], yticks=[], aspect="equal"); ax.invert_yaxis()
ax.legend(markerscale=1.5, fontsize=8, loc="upper left", bbox_to_anchor=(1,1)); plt.tight_layout(); plt.show()

### Exercice 1b: Delaunay graph

k-NN gives every cell exactly `k` neighbours. Delaunay triangulation instead connects cells that are geometrically adjacent, so the number of neighbours adapts to the local layout. It does add a few very long edges across gaps and at the border, so we prune the edges above a length threshold.

1. Build a Delaunay `edge_index`, keeping only the edges shorter than `max_len`. How does its average degree compare to the k-NN graph?

_Hint: `Delaunay(coords).simplices` gives the triangles as triplets of cell indices. Each triangle has 3 edges (column pairs `(0,1)`, `(1,2)`, `(0,2)`). Collect them, deduplicate with `np.unique(np.sort(...), axis=0)`, compute their lengths in `coords`, and drop the long ones._

In [ ]:
from scipy.spatial import Delaunay

max_len = 60      # microns: drop Delaunay edges longer than this

# Exercice 1b: build a pruned Delaunay edge_index
edge_index_delaunay = ...

deg_d = degree(edge_index_delaunay[0], num_nodes=N)
print(f"{edge_index_delaunay.shape[1]} edges, average degree {deg_d.mean():.1f}")

In [ ]:
# k-NN vs Delaunay on the same zoom window
def win_edges(ei):
    s, d = ei.numpy()
    mask = np.array([a in zset and b in zset for a, b in zip(s, d)])
    return coords[ei.numpy().T[mask]]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, ei, title in [(axes[0], edge_index, f"k-NN (k={K_GRAPH})"),
                      (axes[1], edge_index_delaunay, f"Delaunay (pruned at {max_len} um)")]:
    ax.add_collection(LineCollection(win_edges(ei), colors="0.55", linewidths=0.6, alpha=0.7, zorder=0))
    for i in range(D):
        m = zm & (y_domain == i)
        if m.sum(): ax.scatter(coords[m,0], coords[m,1], s=26, color=DOMAIN_COLORS[i], zorder=1, linewidths=0)
    ax.set(title=title, xticks=[], yticks=[], aspect="equal"); ax.invert_yaxis()
plt.tight_layout(); plt.show()

Before pruning, the Delaunay edge lengths have a long tail: most edges are short, but a few cross tissue gaps. That is why we set a threshold.

In [ ]:
tri = Delaunay(coords); s = tri.simplices
e = np.unique(np.sort(np.vstack([s[:,[0,1]], s[:,[1,2]], s[:,[0,2]]]), axis=1), axis=0)
L = np.linalg.norm(coords[e[:,0]] - coords[e[:,1]], axis=1)
plt.figure(figsize=(6, 3.4))
plt.hist(np.clip(L, 0, 120), bins=60, color="0.6")
plt.axvline(max_len, color="crimson", lw=2, label=f"threshold = {max_len} um")
plt.xlabel("Delaunay edge length (um)"); plt.ylabel("count")
plt.title(f"longest edge = {L.max():.0f} um"); plt.legend(); plt.tight_layout(); plt.show()

We build the PyG `Data` object with the k-NN graph and add train / validation / test masks. This is transductive node classification: one graph, and we predict held-out nodes.

> ℹ️ To try the Delaunay graph instead, set `edge_index=edge_index_delaunay` here and rerun the notebook from this cell.

In [ ]:
x = StandardScaler().fit_transform(expr).astype("float32")
data = Data(x=torch.tensor(x), edge_index=edge_index, y=torch.tensor(y_domain, dtype=torch.long))
perm = rng.permutation(N); n_tr, n_va = int(0.6*N), int(0.2*N)
def _mask(idx):
    m = torch.zeros(N, dtype=torch.bool); m[idx] = True; return m
data.train_mask = _mask(perm[:n_tr]); data.val_mask = _mask(perm[n_tr:n_tr+n_va]); data.test_mask = _mask(perm[n_tr+n_va:])
print(data)

## Task 1: predicting the spatial domain (supervised)

We predict each cell's Novae domain from its expression and its position in the graph. We compare two models with the same structure: an MLP that sees each cell on its own, and a GraphSAGE that aggregates the neighbours at each layer. The only difference is `Linear` versus `SAGEConv`.

### Exercice 2: the two models

1. Complete the `MLP` (the baseline; it ignores the graph).
2. Complete the `GraphSAGE` (it passes `edge_index` into each `SAGEConv`).

_Both have two layers and a dropout. Only the layer type changes._

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        # Exercice 2: two linear layers and a dropout
        ...
    def forward(self, x, edge_index):
        # linear -> relu -> dropout -> linear   (edge_index is unused here)
        ...

class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        # Exercice 2: two SAGEConv layers and a dropout
        ...
    def forward(self, x, edge_index):
        # conv1(x, edge_index) -> relu -> dropout -> conv2(x, edge_index)
        ...

### Exercice 3: the training step

1. Fill in the three missing lines: the forward pass, the cross-entropy loss computed on the training nodes only, and the backward pass with the optimizer step.

In [ ]:
def train_model(model, epochs=150, lr=0.01, weight_decay=5e-4):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history, best = {"train": [], "val": []}, {"val": 0.0, "test": 0.0, "state": None}
    for epoch in range(epochs):
        model.train(); opt.zero_grad()
        # Exercice 3: forward pass, loss on the training nodes, backward + step
        out = ...
        loss = ...
        ...

        model.eval()
        with torch.no_grad():
            pred = model(data.x, data.edge_index).argmax(1)
        acc = {k: (pred[m] == data.y[m]).float().mean().item()
               for k, m in [("train", data.train_mask), ("val", data.val_mask), ("test", data.test_mask)]}
        history["train"].append(acc["train"]); history["val"].append(acc["val"])
        if acc["val"] >= best["val"]:
            best = {"val": acc["val"], "test": acc["test"], "state": deepcopy(model.state_dict())}
    return best, history

### Exercice 4: train and compare

1. Instantiate and train both models (`in_dim=n_genes`, `hidden=64`, `out_dim=D`), and print their test accuracies. By how much does the GNN beat the MLP?

In [ ]:
# Exercice 4: instantiate and train both models
torch.manual_seed(SEED); mlp = ...
torch.manual_seed(SEED); gnn = ...
best_mlp, hist_mlp = ...
best_gnn, hist_gnn = ...

print(f"MLP  test accuracy: {best_mlp['test']:.3f}")
print(f"GNN  test accuracy: {best_gnn['test']:.3f}")

The validation curves, and the predictions mapped back onto the tissue. The MLP map is noisy; the GNN map is spatially coherent.

In [ ]:
plt.figure(figsize=(7, 4.3))
plt.plot(hist_mlp["val"], label=f"MLP ({best_mlp['test']:.2f})", color="0.5", lw=2)
plt.plot(hist_gnn["val"], label=f"GraphSAGE ({best_gnn['test']:.2f})", color=DOMAIN_COLORS[0], lw=2)
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
mlp.load_state_dict(best_mlp["state"]); gnn.load_state_dict(best_gnn["state"]); mlp.eval(); gnn.eval()
with torch.no_grad():
    pred_mlp = mlp(data.x, data.edge_index).argmax(1).numpy()
    pred_gnn = gnn(data.x, data.edge_index).argmax(1).numpy()
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
for ax, lab, title in zip(axes, [y_domain, pred_mlp, pred_gnn],
                          ["Novae (truth)", f"MLP ({best_mlp['test']:.2f})", f"GraphSAGE ({best_gnn['test']:.2f})"]):
    for i in range(D):
        m = lab == i; ax.scatter(coords[m,0], coords[m,1], s=4, color=DOMAIN_COLORS[i], linewidths=0)
    ax.set(title=title, xticks=[], yticks=[], aspect="equal"); ax.invert_yaxis()
plt.tight_layout(); plt.show()

### Does the graph matter?

We built two graphs. Let's train the same GraphSAGE on each and compare.

In [ ]:
edge_index_knn = edge_index
for name, ei in [("k-NN", edge_index_knn), ("Delaunay", edge_index_delaunay)]:
    data.edge_index = ei
    torch.manual_seed(SEED); g = GraphSAGE(n_genes, 64, D)
    best, _ = train_model(g)
    print(f"GraphSAGE on {name:9s}: test accuracy {best['test']:.3f}")
data.edge_index = edge_index_knn

A `SAGEConv` layer replaces each cell's features with (roughly) its own features plus the average of its neighbours', then applies a linear transform. Averaging over the neighbours estimates the local cell-type composition, which is essentially what defines a niche. The MLP never gets to compute it.

A few things to try:
- Replace `SAGEConv` with `GCNConv` (already imported).
- Change `K_GRAPH` (2, then 20) and rerun. Too few or too many neighbours both hurt: why?
- Add a third `SAGEConv` layer. Does a deeper GNN keep improving? (look up *over-smoothing*)
- Predict `cell_type` instead of the domain. The gap between the two models should shrink: why?

## Task 2: finding niches without labels (unsupervised)

In practice we usually don't have the domain labels, and finding the niches is the goal. We compare two pipelines, both ending in K-Means, and score them against the Novae domains with the Adjusted Rand Index (ARI: 1 is perfect, 0 is random). We reduce to 50 PCs first to denoise the 422 genes.

### Exercice 5: message passing by hand

Clustering the expression directly (`km_raw`) is given. Your turn to do one message-passing step by hand.

1. Compute `nbr_mean`: replace each cell's features by the average of its neighbours, then cluster it. Compare the two ARIs against the domains and against the cell types. What did the averaging change?

_Hint: `scatter(features[col], row, dim=0, dim_size=N, reduce="mean")` averages, for each node `row`, the features of its neighbours `col`. Apply it to the PCA features `Xpca`._

In [ ]:
Xpca = PCA(n_components=50, random_state=SEED).fit_transform(StandardScaler().fit_transform(expr))

# clustering the expression directly (given)
km_raw = KMeans(D, n_init=10, random_state=SEED).fit_predict(StandardScaler().fit_transform(Xpca))

# neighbourhood graph (k=15, includes each cell itself)
A2 = kneighbors_graph(coords, n_neighbors=15, mode="connectivity", include_self=True).tocoo()
ei2 = to_undirected(torch.tensor(np.vstack([A2.row, A2.col]), dtype=torch.long))
row, col = ei2

# Exercice 5: replace each cell's features by the mean of its neighbours, then cluster
nbr_mean = ...
km_msg = KMeans(D, n_init=10, random_state=SEED).fit_predict(StandardScaler().fit_transform(np.asarray(nbr_mean)))

ari = lambda a, b: adjusted_rand_score(a, b)
print("expression      : ARI vs domains %.3f | vs cell types %.3f" % (ari(y_domain, km_raw), ari(y_celltype, km_raw)))
print("message passing : ARI vs domains %.3f | vs cell types %.3f" % (ari(y_domain, km_msg), ari(y_celltype, km_msg)))

Clustering the expression recovers the cell types. Clustering after one message-passing step recovers the niches. Averaging over space turned the representation from "what a cell is" into "what kind of neighbourhood it sits in".

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
def show(ax, lab, title):
    for i in range(D):
        m = lab == i; ax.scatter(coords[m,0], coords[m,1], s=4, linewidths=0)
    ax.set(title=title, xticks=[], yticks=[], aspect="equal"); ax.invert_yaxis()
show(axes[0], km_raw, "clustering expression")
show(axes[1], km_msg, "clustering after message passing")
for i in range(D):
    m = y_domain == i; axes[2].scatter(coords[m,0], coords[m,1], s=4, color=DOMAIN_COLORS[i], linewidths=0)
axes[2].set(title="Novae domains (truth)", xticks=[], yticks=[], aspect="equal"); axes[2].invert_yaxis()
plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 4))
scores = [ari(y_domain, km_raw), ari(y_domain, km_msg)]
plt.bar(["expression", "message\npassing"], scores, color=["0.6", DOMAIN_COLORS[2]])
for i, s in enumerate(scores): plt.text(i, s + 0.01, f"{s:.2f}", ha="center")
plt.ylabel("ARI vs Novae domains"); plt.ylim(0, max(scores)*1.25); plt.tight_layout(); plt.show()

## Towards learned message passing

We used a single, fixed averaging step. Real methods learn the aggregation with a self-supervised objective: Novae (used in the previous tutorial), and also STAGATE, SpaGCN, CellCharter. Novae's domains are the labels we used here, so we have essentially rebuilt its core idea by hand.

> ℹ️ A graph autoencoder that reconstructs each cell's own expression tends to relearn the cell type, not the niche. Getting niches out of a trained GNN is a question of objective (contrastive learning, or reconstructing the neighbourhood rather than the cell), which is what these tools do.

To go further: run it on another slide (`P1`, `P2`) or the full section, use the Novae `X_scConcept` embeddings as features instead of the raw genes, try `GCNConv` or `GATConv`, and read the GraphSAGE (Hamilton et al. 2017) and Novae papers.